In [25]:
import torch, time
import torch.nn as nn
import torch.nn.functional as F


def conv_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
    )


class CYGNONet(nn.Module):
    def __init__(self, base=16):
        super().__init__()
        # Encoder
        self.down = nn.AvgPool2d(2)
        self.enc1 = conv_block(1,      base)       # 16
        self.enc2 = conv_block(base,   base*2)     # 32
        self.enc3 = conv_block(base*2, base*4)     # 64
        self.enc4 = conv_block(base*4, base*8)     # 128
        self.pool = nn.MaxPool2d(2)
        self.upsample = nn.Upsample(scale_factor=2, mode='nearest')

        # Bottleneck
        self.bottleneck = conv_block(base*8, base*16)  # 256

        # Decoder (lightweight — just upsample + one conv_block)
        self.up4 = nn.ConvTranspose2d(base*16, base*8, 2, stride=2)
        self.dec4 = conv_block(base*16, base*8)

        self.up3 = nn.ConvTranspose2d(base*8, base*4, 2, stride=2)
        self.dec3 = conv_block(base*8,  base*4)

        self.up2 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2)
        self.dec2 = conv_block(base*4,  base*2)

        self.up1 = nn.ConvTranspose2d(base*2, base, 2, stride=2)
        self.dec1 = conv_block(base*2,  base)

        # Single output head
        self.head = nn.Conv2d(base, 1, 1)

    def forward(self, x):
        # Encode
        e1 = self.enc1(self.down(x))
        #e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        # Bottleneck
        b = self.bottleneck(self.pool(e4))

        # Decode with skips
        d4 = self.dec4(torch.cat([self.up4(b),  e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.upsample(self.head(d1))  # raw logits, (N, 1, H, W)

def focal_loss(pred, target, alpha=0.25, gamma=2.0):
    # target: binary float mask from redpixels > 0
    bce = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
    pt  = torch.exp(-bce)
    return (alpha * (1 - pt)**gamma * bce).mean()

def kl_loss(y_actual, x_pred, eps=1e-8):
    xx = x_pred.flatten()
    yy = y_actual.flatten()
    return ((yy - xx) * torch.log(yy / (xx + eps))).sum() / xx.numel()

def loss_alias(pred, target):
    return kl_loss(pred, target)

b_size = 6
model = CYGNONet()
device = torch.device("cuda")
dummy = torch.randn(b_size, 1, 2304, 4096, dtype=torch.float16, device=device)
model = model.to(device).half()
print(model)

torch.cuda.reset_peak_memory_stats()
pred = model(dummy)
loss = loss_alias(pred, dummy)
loss.backward()
peak = torch.cuda.max_memory_allocated() / 1e9
print(f"Peak VRAM with batch size {b_size}: {peak:.1f} GB")

dummy = torch.randn(1, 1, 2304, 4096, dtype=torch.float16, device=device)

model.eval()
# Warmup
for _ in range(10):
    _ = model(dummy)

torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(10):
    with torch.inference_mode():
        _ = model(dummy)
torch.cuda.synchronize()
tim = (time.perf_counter() - t0) / 10 * 1000
print(f"{tim:.1f} ms per frame")

CYGNONet(
  (down): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (enc1): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (4): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
  )
  (enc2): Sequential(
    (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
  )
  (enc3): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), s